# Find the manual ground-truth best match per NDS event

For each of the 4 NDS events, run targeted SQL queries to find the operation rows in the corpus that best fit the event description by literal keyword overlap. Use the results to:

- Validate whether the matcher's top-1 picks (TF-IDF / SBERT / hybrid / reranked) are correct
- Build the slide-deck comparison table
- Identify which method works best per event

The 4 NDS events:

| # | Well | Event |
|---|---|---|
| 1 | F-10 | inclination angle was higher than expected. Reduced inclination to 0.16 deg |
| 2 | F-11 | tight hole event was encountered while drilling interval at 958 m |
| 3 | F-12 | Excessive clay accumulation observed in BHA while POOH 26" BHA |
| 4 | F-13 | differential stuck during RIH with 20" casing, mud replaced with seawater, 300 m³ pumped, set-down weight increased, string came free |

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.common.db import get_connection
pd.set_option("display.max_colwidth", 250)
pd.set_option("display.width", 240)

## Event 1 — F-10: inclination higher than expected, reduced to 0.16 deg

Look for F-10 ops that mention inclination + reduction + the specific 0.16 value, OR survey operations with high-inclination corrections.

In [2]:
with get_connection(read_only=True) as con:
    df = con.execute("""
        SELECT report_date, start_time, end_time, end_depth_md, main_activity, sub_activity, state, remark
        FROM operations
        WHERE well_family = '15_9_F_10'
          AND remark IS NOT NULL
          AND (
                remark ILIKE '%inclination%'
             OR remark ILIKE '%incl.%'
             OR remark ILIKE '%0.16%'
          )
        ORDER BY report_date, start_time
        LIMIT 20
    """).df()
df

,report_date,start_time,end_time,end_depth_md,main_activity,sub_activity,state,remark
0,2009-04-07,16:00,19:00,151.0,drilling,drill,ok,"Stopped rotation and pumps. ROV moved into template to observe position of centralizer drill bushing. Continued to drill from 148,1 m to 151 m. 1000 lpm, 20 RPM. 19:00:00 165 drilling -- drill ok Drilled 36"" hole from 151 m to 165 m. 80 RPM, 4 -5..."
1,2009-04-08,00:00,05:00,207.0,drilling,drill,ok,"Drilled 36"" hole from 165 m to 207 m (17 1/2"" bit depth). 90-100 RPM, 5 ton WOB, 4000-4450 lpm, 3,7 kNm, ROP 5-20 m/hr. Pumped HIVIS pills as per progra m. Took survey at 170 m: Inclination 0,27 deg. Took survey at 180 m: Inclination 0,17 deg. To..."
2,2009-04-12,01:45,03:45,228.0,drilling --,NaN,ok,"Drilled 26"" hole from 217 m to 228 m MD. Drilling parameters : Flow 3500 lpm / SPP 48-95 bar / 40 RPM / WOB < 1 MT / Torque 2-3 kNm. Observed erratic string and torque peaks of drill 6-7 kNm at 227 m MD. Secured string on Centralizer deck. Perfor..."
3,2009-04-12,04:15,05:00,233.0,drilling --,NaN,ok,"Drilled 26"" hole from 231 m to 233 m MD. Drilling parameters : Flow 3500 lpm / SPP 100 bar / 40 RPM / WOB 1-2 MT / Torque 2-3 kNm. Secured string on Centralizer deck. Performe drill d gMWD survey - out of specification. MWD survey inclination 0,2..."
4,2009-04-12,07:15,07:30,245.0,drilling --,NaN,ok,Reamed interval 230-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 97 bar / 40 RPM / Torque 2-4 kNm. drill
5,2009-04-12,09:00,11:15,246.0,drilling --,NaN,ok,"Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pills. Reduced iclinatio drill n from 0,53 deg to 0,16 deg."


In [3]:
# Narrower: explicit reduction language
with get_connection(read_only=True) as con:
    df = con.execute("""
        SELECT report_date, start_time, end_depth_md, remark
        FROM operations
        WHERE well_family = '15_9_F_10'
          AND remark IS NOT NULL
          AND remark ILIKE '%inclination%'
          AND (remark ILIKE '%reduce%' OR remark ILIKE '%reduc%' OR remark ILIKE '%lower%' OR remark ILIKE '%drop%')
        ORDER BY report_date
    """).df()
print(f"Found {len(df)} candidate rows for Event 1")
df

Found 2 candidate rows for Event 1


,report_date,start_time,end_depth_md,remark
0,2009-04-12,07:15,245.0,Reamed interval 230-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 97 bar / 40 RPM / Torque 2-4 kNm. drill
1,2009-04-12,09:00,246.0,"Reamed interval 222-246 m MD in order to reduce inclination. Parameters : Flow 3500 lpm / SPP 95 bar / 140 RPM / Torque 2-4 kNm. Pumped 2 x 10 m3 hivis pills. Reduced iclinatio drill n from 0,53 deg to 0,16 deg."


## Event 2 — F-11: tight hole encountered while drilling at 958 m

Search the F-11 family (F_11, F_11_A, F_11_B, F_11_T2) for tight-hole mentions near 958 m.

In [4]:
with get_connection(read_only=True) as con:
    df = con.execute("""
        SELECT well_prefix, report_date, start_time, end_depth_md, main_activity, sub_activity, state, remark
        FROM operations
        WHERE well_family = '15_9_F_11'
          AND remark IS NOT NULL
          AND remark ILIKE '%tight%'
          AND (
                remark ILIKE '%958%'
             OR remark ILIKE '%hole%'
          )
        ORDER BY report_date, start_time
        LIMIT 30
    """).df()
print(f"Found {len(df)} candidate rows for Event 2")
df

Found 4 candidate rows for Event 2


,well_prefix,report_date,start_time,end_depth_md,main_activity,sub_activity,state,remark
0,15_9_F_11_T2,2013-04-02,12:00,460.0,drilling,trip,ok,"Pulled out of hole with 26"" bottom hole assembly from 1365m MD to 460m MD. Observed tight hole at 460m MD. Attempted to get past restriction with maximum 30MT overpull - no go."
1,15_9_F_11_T2,2013-05-09,17:45,4407.0,drilling,trip,ok,"Pulled out of hole with 8 1/2"" bottom hole assembly from 4562m MD to 4467m MD. Took 5MT overpull at 4467m MD. Attempted to pass restriction several times with maximum 30MT - no go. Attempted to run in hole with 8 1/2"" bottom hole assembly with up..."
2,15_9_F_11_T2,2013-05-10,00:00,2550.0,drilling,trip,ok,"Pulled / lubricated out of hole with 8 1/2"" bottom hole assembly from 4266m MD to 2550m MD with 200 lpm and 8-9 bar. Observed tight spots at 4230m, 4225m, 4066m-4060m and 3950m with maximum 20 MT overpull."
3,15_9_F_11_B,2013-06-15,21:00,4095.0,drilling,casing,ok,"From 3856m MD, observed frequent tight spots corresponding with high doglegs. Ran in hole with 7"" liner on 5 1/2"" landing string and worked through tight spots from 3856m MD to 4095m MD. Average tripping speed 119 m/hr. No losses to formation. 23..."


In [5]:
# Narrowest: "tight hole" near 958 m
with get_connection(read_only=True) as con:
    df = con.execute("""
        SELECT well_prefix, report_date, start_time, end_depth_md, remark
        FROM operations
        WHERE well_family = '15_9_F_11'
          AND remark IS NOT NULL
          AND remark ILIKE '%tight hole%'
          AND remark ILIKE '%958%'
        ORDER BY report_date
    """).df()
df

,well_prefix,report_date,start_time,end_depth_md,remark


In [6]:
# Backup: any F-11 op at depth ~958 m
with get_connection(read_only=True) as con:
    df = con.execute("""
        SELECT well_prefix, report_date, start_time, end_depth_md, remark
        FROM operations
        WHERE well_family = '15_9_F_11'
          AND remark IS NOT NULL
          AND end_depth_md BETWEEN 950 AND 970
          AND (remark ILIKE '%tight%' OR remark ILIKE '%stuck%' OR remark ILIKE '%pack%')
        ORDER BY report_date
    """).df()
df

,well_prefix,report_date,start_time,end_depth_md,remark


## Event 3 — F-12: clay accumulation in BHA while POOH 26" BHA

We already analyzed this one. The truth is `2007_07_12.pdf` (POOH 26" BHA with overpull). Reproduce here for completeness.

In [7]:
with get_connection(read_only=True) as con:
    df = con.execute("""
        SELECT report_date, start_time, end_depth_md, remark
        FROM operations
        WHERE well_family = '15_9_F_12'
          AND remark IS NOT NULL
          AND remark LIKE '%26"%'
          AND remark ILIKE '%POOH%'
        ORDER BY report_date
    """).df()
print(f"F-12 ops mentioning 26\" + POOH: {len(df)}")
df

F-12 ops mentioning 26" + POOH: 5


,report_date,start_time,end_depth_md,remark
0,2007-06-23,03:00,29.0,"Continued POOH with 26"" HO BHA and racked back same."
1,2007-06-23,05:30,29.0,"Continued POOH with 26"" HO BHA and racked back same."
2,2007-06-23,06:00,0.0,"POOH with 26"" HO BHA from 29 m to 8 m and racked back same. Held tool box meeting prior to POOH with 12 1/4"" x 17 1/2"" x 26"" hole opener assembly. Removed mast er bushing and LO 26"" HO BHA. Cleared rig floor."
3,2007-07-12,02:00,1002.0,"POOH w/ 26"" BHA. F/ 1369m T/ 1002m. Lubricated out due to over pull of 20 MT. F/ 1340m T/ 1303m and F/1102m T/1002. 442 lpm, 2.5 bars, 80 rpm, 8 - 20 KNm."
4,2007-07-12,06:00,870.0,"POOH w/ 26"" BHA. Lubricated F/ 1002m T/ 870m. 442 lpm, 2.5 bars, 80 rpm, 8 - 20 KNm."


In [8]:
# Strongest: 26" + POOH + accumulation/clay/pack/overpull/tight
with get_connection(read_only=True) as con:
    df = con.execute("""
        SELECT report_date, start_time, end_depth_md, remark
        FROM operations
        WHERE well_family = '15_9_F_12'
          AND remark IS NOT NULL
          AND remark LIKE '%26"%'
          AND remark ILIKE '%POOH%'
          AND (
                remark ILIKE '%clay%'
             OR remark ILIKE '%accumul%'
             OR remark ILIKE '%pack%'
             OR remark ILIKE '%overpull%'
             OR remark ILIKE '%over pull%'
             OR remark ILIKE '%stick%'
             OR remark ILIKE '%tight%'
          )
        ORDER BY report_date
    """).df()
df

,report_date,start_time,end_depth_md,remark
0,2007-07-12,02:00,1002.0,"POOH w/ 26"" BHA. F/ 1369m T/ 1002m. Lubricated out due to over pull of 20 MT. F/ 1340m T/ 1303m and F/1102m T/1002. 442 lpm, 2.5 bars, 80 rpm, 8 - 20 KNm."


## Event 4 — F-13: differential stuck while RIH with 20" casing

F-13 has no PDFs in the corpus. Search the WHOLE corpus for: differential stuck + RIH + 20" + casing + seawater (mud replacement) + 300 m³.

In [9]:
# Strongest: differential stuck + 20" casing + seawater
with get_connection(read_only=True) as con:
    df = con.execute("""
        SELECT well_family, well_prefix, report_date, start_time, end_depth_md, remark
        FROM operations
        WHERE remark IS NOT NULL
          AND remark ILIKE '%differential%'
          AND remark ILIKE '%stuck%'
          AND remark LIKE '%20"%'
        ORDER BY report_date
        LIMIT 20
    """).df()
print(f"Found {len(df)} ops with differential stuck + 20\"")
df

Found 1 ops with differential stuck + 20"


,well_family,well_prefix,report_date,start_time,end_depth_md,remark
0,15_9_F_10,15_9_F_10,2009-04-18,07:30,1238.0,"Attempted to RIH with 20"" casing / 18 3/4"" WH - negative. Worked to free pipe with alternating up/down weights, 50 MT set down / 60 MT overpull - negative - 20"" casing differential stuck. n -- fish String movement < 50 cm. Set down 50 MT weight. ..."


In [10]:
# Looser: any differential stuck mention near a 20" casing/RIH context
with get_connection(read_only=True) as con:
    df = con.execute("""
        SELECT well_family, report_date, start_time, end_depth_md, remark
        FROM operations
        WHERE remark IS NOT NULL
          AND remark ILIKE '%differential stuck%'
        ORDER BY report_date
        LIMIT 20
    """).df()
print(f"Found {len(df)} ops with literal 'differential stuck'")
df

Found 1 ops with literal 'differential stuck'


,well_family,report_date,start_time,end_depth_md,remark
0,15_9_F_10,2009-04-18,07:30,1238.0,"Attempted to RIH with 20"" casing / 18 3/4"" WH - negative. Worked to free pipe with alternating up/down weights, 50 MT set down / 60 MT overpull - negative - 20"" casing differential stuck. n -- fish String movement < 50 cm. Set down 50 MT weight. ..."


In [11]:
# Even looser: stuck + casing + seawater (mud replacement)
with get_connection(read_only=True) as con:
    df = con.execute("""
        SELECT well_family, report_date, start_time, end_depth_md, remark
        FROM operations
        WHERE remark IS NOT NULL
          AND remark ILIKE '%stuck%'
          AND remark ILIKE '%casing%'
          AND (remark ILIKE '%seawater%' OR remark ILIKE '%sea water%' OR remark ILIKE '%SW%')
        ORDER BY report_date
        LIMIT 20
    """).df()
df

,well_family,report_date,start_time,end_depth_md,remark
0,15_9_F_10,2009-04-18,07:30,1238.0,"Attempted to RIH with 20"" casing / 18 3/4"" WH - negative. Worked to free pipe with alternating up/down weights, 50 MT set down / 60 MT overpull - negative - 20"" casing differential stuck. n -- fish String movement < 50 cm. Set down 50 MT weight. ..."


In [12]:
# Most specific signature: "300 m3" of seawater pumped (very distinctive number)
with get_connection(read_only=True) as con:
    df = con.execute("""
        SELECT well_family, report_date, start_time, remark
        FROM operations
        WHERE remark IS NOT NULL
          AND (remark ILIKE '%300 m3%' OR remark ILIKE '%300m3%')
          AND (remark ILIKE '%seawater%' OR remark ILIKE '%sea water%')
        ORDER BY report_date
    """).df()
df

,well_family,report_date,start_time,remark
0,15_9_F_10,2009-04-18,11:15,Attempted to free string while circulating at 2000 lpm. Set down / overpull 60/60 MT. Increased set down weight to 80 MT and observed string comming free. Took up/down weights 270/22 n -- fish 0 MT. Cummulative volume of seawater pumped ~300 m3.


## Summary table — fill in after running the queries above

Once you identify the most likely true best match for each event, write it down here. This becomes your slide-deck reference table.

In [13]:
ground_truth = pd.DataFrame([
    {"event_id": 1, "nds_well": "15/9-F-10", "true_pdf": "FILL_IN", "true_op_id": None,
     "reasoning": "Look for inclination reduction matching 0.16 deg"},
    {"event_id": 2, "nds_well": "15/9-F-11", "true_pdf": "FILL_IN", "true_op_id": None,
     "reasoning": "Look for tight-hole event near 958 m"},
    {"event_id": 3, "nds_well": "15/9-F-12", "true_pdf": "15_9_F_12_2007_07_12.pdf", "true_op_id": None,
     "reasoning": "POOH w/ 26\" BHA with 20 MT overpull (the literal+semantic best match)"},
    {"event_id": 4, "nds_well": "15/9-F-13", "true_pdf": "FILL_IN (cross-well)", "true_op_id": None,
     "reasoning": "F-13 has no PDFs; find best cross-well evidence"},
])
ground_truth

,event_id,nds_well,true_pdf,true_op_id,reasoning
0,1,15/9-F-10,FILL_IN,None,Look for inclination reduction matching 0.16 deg
1,2,15/9-F-11,FILL_IN,None,Look for tight-hole event near 958 m
2,3,15/9-F-12,15_9_F_12_2007_07_12.pdf,None,"POOH w/ 26"" BHA with 20 MT overpull (the literal+semantic best match)"
3,4,15/9-F-13,FILL_IN (cross-well),None,F-13 has no PDFs; find best cross-well evidence
